In [10]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from forecast_checks import ForecastValidator, MakeTrainingData, DELAY_DAYS

In [11]:
trainer = MakeTrainingData()
data = trainer.get_train_test()
train_split = data["train"]
test_split=data["test"]
X_train, y_train = train_split["feature"], train_split["target"]
X_test, y_test = test_split["feature"], test_split["target"]

In [12]:
#Score every model on the same hours: drop test rows where the baseline has no value
scored = X_test["consumption_7days_ago"].notna()
X_test, y_test = X_test[scored], y_test[scored]

In [ ]:
#Hyperparameter tuning with CVGrid

NUM_TREES=[100,500,1000]
LEARNING_RATE=[0.025, 0.05, 0.075]
MAX_DEPTH=[5,10,15]

BASE_PARAMS = dict(enable_categorical=True, tree_method="hist", random_state=42)
param_grid = {"n_estimators": NUM_TREES, "learning_rate": LEARNING_RATE, "max_depth": MAX_DEPTH}
#Rows are sorted by time; the gap leaves out ~6 days of rows between fit and validation, like the consumption delay
cv = TimeSeriesSplit(n_splits=3, gap=(DELAY_DAYS + 1) * 24 * X_train["location"].nunique())
search = GridSearchCV(xgb.XGBRegressor(**BASE_PARAMS), param_grid, cv=cv,
                      scoring="neg_mean_absolute_error", refit=False,
                      verbose=True)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)

Fitting 3 folds for each of 27 candidates, totalling 81 fits


In [ ]:
#TRAIN

model = xgb.XGBRegressor(**BASE_PARAMS, **search.best_params_)
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [ ]:
#TEST

forecastValidator = (ForecastValidator(y_test, groups=X_test["location"])
                     .add("Same hour last week", X_test["consumption_7days_ago"], baseline=True)
                     .add("XGBoost", pred))
forecastValidator.summary()

In [ ]:
forecastValidator.save_to_json("results/xgboost.json")

In [ ]:
forecastValidator.per_group()

In [ ]:
forecastValidator.plot_predicted_vs_actual_consumption(no_baseline=True)

In [ ]:
forecastValidator.plot_consumption_over_time(trainer.df.loc[y_test.index, "time"], no_baseline=True)